# DesignJanus: Final Analysis

Answers:
1. Which failure types hurt overall consistency most?
2. Which hurt design usefulness most?
3. Which object categories are hardest?
4. Which automatic metric best correlates with human judgment?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

REPO = Path("..")
MERGED = REPO / "data" / "annotations" / "merged_results.csv"

df = pd.read_csv(MERGED)
print(df.shape)
df.head()

## 1. Which failure types hurt overall consistency most?

In [ ]:
dims = ["identity", "geometry", "parts", "material", "silhouette"]
for d in dims:
    if d in df.columns:
        r = df[d].corr(df["overall"])
        print(f"{d} vs overall: r = {r:.3f}")

# Plot: per-dimension mean rating vs overall
fig, ax = plt.subplots(figsize=(8, 4))
means = df.groupby("overall")[dims].mean().mean(axis=1)
means.plot(kind="bar", ax=ax)
ax.set_title("Mean per-category rating by overall consistency")
plt.tight_layout()
plt.show()

## 2. Which hurt design usefulness most?

In [ ]:
if "usefulness" in df.columns:
    for d in dims:
        if d in df.columns:
            r = df[d].corr(df["usefulness"])
            print(f"{d} vs usefulness: r = {r:.3f}")

## 3. Which object categories are hardest?

In [ ]:
if "category" in df.columns:
    by_cat = df.groupby("category")["overall"].agg(["mean", "std", "count"])
    by_cat = by_cat.sort_values("mean")
    display(by_cat)
    by_cat["mean"].plot(kind="barh", figsize=(6, 4))
    plt.title("Mean overall consistency by category")
    plt.tight_layout()
    plt.show()

## 4. Which automatic metric best correlates with human judgment?

In [ ]:
metric_cols = ["clip_adjacent_sim", "silhouette_overlap", "color_variance", "dino_adjacent_sim"]
metric_cols = [c for c in metric_cols if c in df.columns]

for m in metric_cols:
    valid = df[[m, "overall"]].dropna()
    if len(valid) > 5:
        r = valid[m].corr(valid["overall"])
        print(f"{m} vs overall: r = {r:.3f}")

# Scatter: best metric vs overall
if metric_cols:
    best = metric_cols[0]
    valid = df[[best, "overall"]].dropna()
    plt.figure(figsize=(5, 4))
    plt.scatter(valid[best], valid["overall"])
    plt.xlabel(best)
    plt.ylabel("overall")
    plt.title("Automatic metric vs human overall")
    plt.tight_layout()
    plt.show()

## Summary: Key insight

In [ ]:
# Fill in after analysis:
# "Semantic-part inconsistency and silhouette drift mattered more to perceived
# design usefulness than mild material or lighting inconsistency."